In [1]:
import sys
# !{sys.executable} -m pip install monai --quiet
# !{sys.executable} -m pip install pandas --quiet
# !{sys.executable} -m pip install wandb --quiet
# !{sys.executable} -m pip install scikit-learn --quiet
# !{sys.executable} -m pip install einops --quiet
# !{sys.executable} -m pip install torchsummary --quiet
# !{sys.executable} -m pip install timm --quiet
# !{sys.executable} -m pip install albumentations --quiet
# !{sys.executable} -m pip install matplotlib --quiet
# !{sys.executable} -m pip install nibabel --quiet
# !{sys.executable} -m pip install nnunet --no-deps --quiet
# !{sys.executable} -m pip install tifffile  


In [ ]:
import time
import os
import torch
import pandas as pd
# from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torch.nn.functional as F

from scipy import ndimage
from glob import glob
import pandas as pd
from pathlib import Path

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")



from random import sample
import nibabel as nib

plt.ion()   # interactive mode

In [3]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from pathlib import Path
import pandas as pd
import nibabel as nib
import numpy as np
from tifffile import imread
from tqdm import tqdm
import json
import os

# --- INPUT DATA ---
DATA_DIR = Path("data/")
CSV_PATH = DATA_DIR / "train.csv"
IMG_DIR = DATA_DIR / "train_images"
LBL_DIR = DATA_DIR / "train_labels"
REPO_DIR = 'nnunet_repo'
# --- OUTPUT NNUNET DIR ---
BASE = Path("/kaggle/ft_nnunet/nnUNet/nnunet/nnUNet_raw_data_base/nnUNet_raw")
task_id = 900
task_name = "VesuviusScroll"
task_folder = f"Dataset{task_id:03d}_{task_name}"

base_dir = BASE / task_folder
imagesTr = base_dir / "imagesTr"
labelsTr = base_dir / "labelsTr"

# make folders
imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)


In [5]:
base_dir.mkdir(parents=True, exist_ok=True)
imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)
# test_dir.mkdir(parents=True, exist_ok=True)
# trained_model_dir.mkdir(parents=True, exist_ok=True)

In [6]:
from PIL import Image, ImageSequence

def safe_tiff_read(path):
    """Fast multi-page TIFF reader using PIL (handles LZW)."""
    img = Image.open(path)

    # read all pages efficiently
    frames = [np.array(frame) for frame in ImageSequence.Iterator(img)]

    # stack into (Z, H, W) or (Z, H, W, C)
    return np.stack(frames, axis=0)

In [8]:
import shutil
SPACING = [1, 1, 1]  # change if needed

df = pd.read_csv(CSV_PATH)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Copying TIFFs"):
    case_id = str(row["id"])
    scroll_id = str(row["scroll_id"])

    img_path = IMG_DIR / f"{case_id}.tif"
    lbl_path = LBL_DIR / f"{case_id}.tif"

    if not img_path.exists() or not lbl_path.exists():
        print(f"Skipping missing pair: {scroll_id}")
        continue

    # destination names for nnUNet
    img_dst = imagesTr / f"{case_id}_0000.tif"
    lbl_dst = labelsTr / f"{case_id}.tif"

    json_dst_img = imagesTr / f"{case_id}.json"
    json_dst_lbl = labelsTr / f"{case_id}.json"

    # ---- COPY FILES (FAST) ----
    shutil.copy2(img_path, img_dst)
    shutil.copy2(lbl_path, lbl_dst)

    # ---- WRITE SPACING JSON ----
    spacing_info = {"spacing": SPACING}

    with open(json_dst_img, "w") as f:
        json.dump(spacing_info, f)

    with open(json_dst_lbl, "w") as f:
        json.dump(spacing_info, f)

Copying TIFFs: 100%|██████████| 806/806 [01:45<00:00,  7.66it/s]


In [9]:
import os

os.environ["nnUNet_raw"] = "/kaggle/ft_nnunet/nnUNet/nnunet/nnUNet_raw_data_base/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/ft_nnunet/nnUNet/nnunet/preprocessed"
os.environ["nnUNet_results"] = "/kaggle/ft_nnunet/nnUNet/nnunet/nnUNet_results"

In [ ]:
dataset_json = {
  "name": "Vesuvius Scroll Surface Detection",
  "description": "Binary segmentation of ink regions in 3D X-ray micro-CT TIFF volumes.",
  "reference": "",
  "licence": "",
  "release": "0.1",
  "tensorImageSize": "3D",

  "channel_names": {
    "0": "CT"
  },

  "labels": {
     "background" : 0,
     "surface" : 1,
     "ignore" : 2
  },

  "file_ending": ".tif",

  "numTraining": 806,


}

with open(base_dir / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=4)

print("✔️ dataset.json written to:", base_dir / "dataset.json")

✔️ dataset.json written to: /kaggle/ft_nnunet/nnUNet/nnunet/nnUNet_raw_data_base/nnUNet_raw/Dataset900_VesuviusScroll/dataset.json


In [12]:
%cd $REPO_DIR 
# !pip install -q --upgrade pip
!{sys.executable} -m pip install   git+https://github.com/MIC-DKFZ/nnUNet.git

/kaggle/ft_nnunet
  Cloning https://github.com/MIC-DKFZ/nnUNet.git to /tmp/pip-req-build-bpqc06_3
  Running command git clone --filter=blob:none --quiet https://github.com/MIC-DKFZ/nnUNet.git /tmp/pip-req-build-bpqc06_3
  Resolved https://github.com/MIC-DKFZ/nnUNet.git to commit 86606c53ef9f556d6f024a304b52a48378453641
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# %cd $REPO_DIR 
sys.path.append(REPO_DIR)
!{sys.executable} -m nnunetv2.experiment_planning.plan_and_preprocess_entrypoints -d 900 --verify_dataset_integrity 

Fingerprint extraction...
Dataset900_VesuviusScroll
Failed to open file /kaggle/ft_nnunet/nnUNet/nnunet/nnUNet_raw_data_base/nnUNet_raw/Dataset900_VesuviusScroll/imagesTr/1004283650_0000.tif with reader <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'>:
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/site-packages/nnunetv2/imageio/reader_writer_registry.py", line 49, in determine_reader_writer_from_file_ending
    _ = tmp.read_images((example_file,))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/nnunetv2/imageio/natural_image_reader_writer.py", line 43, in read_images
    assert npy_img.shape[-1] == 3 or npy_img.shape[-1] == 4, "If image has three dimensions then the last " \
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: If image has three dimensions then the last dimension must have shape 3 or 4 (RGB or RGBA). Image shape here is (320, 320, 320)
Using <class 'nnunetv2.im

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="fft_conv_pytorch")

In [ ]:
!{sys.executable} -m nnunetv2.run.run_training 552 3d_fullres 0 -num_gpus 2 

In [ ]:
!{sys.executable} -m nnunetv2.run.run_training 552 3d_fullres 1 -num_gpus 2 